# Notebook 084: Vectorized Execution and Photon Engine Acceleration Mechanics

**Module:** Module 08: Performance Tuning, Query Optimization & Skew Mitigation  
**Difficulty:** `Advanced`  
**Databricks Masterclass Reference:** Video 02: Databricks Intro, Video 13: Databricks Clusters & Video 25: Genie Code  

---

## 1. Business Scenario & Real-World Context
An enterprise evaluates migrating high-throughput SQL analytics from classic Apache Spark clusters to Databricks clusters with Photon Engine enabled. Leadership requires an architectural analysis of C++ vectorized execution vs JVM row-based execution.

## 2. Technical Requirement & Problem Statement
Deep dive into Vectorized Execution and Databricks Photon: understand CPU cache locality, SIMD vectorization, columnar in-memory processing, identify operators supported by Photon, and analyze execution speedup profiles.

## 3. Spark & Databricks Architecture Notes
Classic Apache Spark compiles queries into Java bytecode running inside JVMs. Databricks Photon is a vectorized execution engine written from scratch in C++. It operates on columnar batches in CPU L1/L2 caches and uses SIMD (Single Instruction, Multiple Data) instructions, accelerating SQL and aggregation workloads by 2x to 5x.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_084_Vectorized_Execution_and_Photon_Engine_Acceleration_Mechanics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Architectural Comparison: Classic JVM Spark vs Databricks Photon
Analyze the fundamental architectural differences between engines.

In [ ]:
comparison = [
    ("Core Language", "Java / Scala (JVM Bytecode)", "C++ (Native Machine Code)"),
    ("Execution Model", "Tungsten Whole-Stage CodeGen", "Vectorized Columnar Batch Processing"),
    ("CPU Optimization", "Standard JVM register usage", "SIMD (Single Instruction Multiple Data)"),
    ("Memory Management", "JVM Garbage Collection & Off-Heap", "Direct C++ Memory Management (Zero GC overhead)"),
    ("Supported Workloads", "General PySpark, RDDs, Custom UDFs", "SQL, Aggregations, Joins, Parquet/Delta I/O")
]

schema = "dimension STRING, classic_spark STRING, databricks_photon STRING"
df_comp = spark.createDataFrame(comparison, schema)
df_comp.show(truncate=False)

## 6. Benchmarking Vector-Friendly Aggregations in PySpark
Construct queries optimized for vectorized batch execution.

In [ ]:
import time

# Vectorized engines excel at high-speed columnar filtering and math
df_vec_bench = spark.range(1, 1000000, 1, 4).select(
    F.col("id"),
    (F.col("id") * 1.5).alias("price"),
    (F.col("id") % 5).alias("category")
)

t0 = time.time()
res = df_vec_bench.filter(F.col("price") > 500000).groupBy("category").agg(F.sum("price"), F.avg("price")).collect()
elapsed = time.time() - t0

print(f"Columnar Vector-Friendly Aggregation evaluated in {elapsed:.3f}s")

## 7. Photon Compatibility Best Practices
Identify anti-patterns that force Photon to fall back to classic JVM execution.

In [ ]:
rules = [
    ("Native Built-ins", "Always use native Spark SQL functions; Photon accelerates 100% of built-in expressions."),
    ("Avoid Non-Vectorized UDFs", "Standard Python @udf forces Photon to serialize data back into Python subprocesses!"),
    ("Parquet / Delta Storage", "Photon reads directly from Parquet and Delta columnar formats with zero-copy decoding.")
]

for title, rule in rules:
    print(f" {title}: {rule}")

## 9. Key Architectural Takeaways
- **Photon is Databricks' vectorized query engine written in C++ for maximum CPU cache locality and SIMD execution.**
- **Photon eliminates Java Virtual Machine garbage collection pauses entirely for data operations.**
- **Custom Python UDFs disable Photon acceleration for that stage, forcing fallback to classic Spark JVM.**

## 10. Senior Data Engineering Interview Questions
1. What is Databricks Photon and how does it differ from classic Apache Spark Tungsten Whole-Stage CodeGen?
2. What is SIMD (Single Instruction, Multiple Data) and how does vectorized execution leverage it?
3. Which query operations benefit most from Photon, and which operations force a fallback to the JVM?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
